# Nhập run train trên máy GPU riêng vào `runs_aug1`

1. Tải các file `aug1_*.zip` từ `pc_runs/upload/` của máy riêng lên `MyDrive/Sym/upload/`.
2. Chọn runtime **CPU** là đủ. Trong 🔑 Secrets bật **Notebook access** cho `GH_TOKEN`.
3. Chạy **Setup** (đã đặt `IMAGES = False`, không giải nén ảnh), rồi **cell I**.

Cell I không ghi đè run đã có trên Drive và không đụng test. Zip bị từ chối nằm ở `upload/rejected/`: gửi Claude dòng thông báo của zip đó.

In [ ]:
import os, sys, json, time, base64, shutil, subprocess
from pathlib import Path
from google.colab import drive, userdata

if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/Letri6102/Thesis_Sym.git'
BRANCH   = 'main'                      # nhánh stage1-ema-valchecks đã merge vào main (21/11)
REPO     = Path('/content/Thesis_Sym')
CODE     = REPO / 'Contrastive_V1'
SYM      = Path('/content/drive/MyDrive/Sym')
SPLITS   = SYM / 'splits_v2'           # manifest mới; splits/ cũ chỉ cho runs_retry1
RUNS     = SYM / 'runs_aug1'           # run mới với strong_train_aug
LOGS     = SYM / 'logs'
IMAGES   = False                       # False: không giải nén ảnh (phiên CPU chỉ chạy Tầng 2, cell E)

tok  = userdata.get('GH_TOKEN')
auth = 'http.extraHeader=Authorization: Basic ' + base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
def git(*args, cwd=None):
    r = subprocess.run(['git', '-c', auth, *args], cwd=cwd, capture_output=True, text=True,
                       env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
    if r.returncode:
        raise RuntimeError(f'git {args[0]} lỗi (exit {r.returncode}): {r.stderr.strip()}')
try:
    if (REPO / '.git').exists():
        # refspec đầy đủ: bản clone 1 nhánh vẫn đổi được sang nhánh khác (fetch tên trơn chỉ cập nhật FETCH_HEAD)
        git('fetch', '-q', '--depth', '1', 'origin', f'+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}', cwd=REPO)
        git('reset', '-q', '--hard', f'origin/{BRANCH}', cwd=REPO)
    else:
        git('clone', '-q', '--depth', '1', '-b', BRANCH, REPO_URL, str(REPO))
finally:
    del tok, auth
print('commit:', subprocess.run(['git', 'log', '-1', '--format=%h %s'], cwd=REPO,
                                capture_output=True, text=True).stdout.strip())

SHARDS = SYM / 'DISFA_img'                     # 27 tar theo subject, do cell A tạo
if IMAGES and (SHARDS / 'DONE.json').exists():
    DATA = Path('/content/DISFA')
    if not (DATA / '.img_ok').exists():        # chưa giải nén, hoặc lần trước giải nén dở
        t0, meta = time.time(), json.loads((SHARDS / 'DONE.json').read_text())
        DATA.mkdir(parents=True, exist_ok=True)
        print(f'Giải nén {len(meta["subjects"])} tar ({meta["bytes"] / 1e9:.1f} GB) về /content:', flush=True)
        for i, s in enumerate(meta['subjects'], 1):
            subprocess.run(['tar', '-xf', str(SHARDS / f'{s}.tar'), '-C', str(DATA)], check=True, timeout=1800)
            print(f'  {i:>2}/{len(meta["subjects"])} {s}  {(time.time() - t0) / 60:.1f} phút', flush=True)
        n = sum(len(f) for _, _, f in os.walk(DATA / 'img'))
        assert n == meta['n_images'], f'Giải nén được {n} ảnh, cần {meta["n_images"]}'
        shutil.copytree(SYM / 'DISFA' / 'list', DATA / 'list', dirs_exist_ok=True)
        (DATA / '.img_ok').touch()
        print(f'Đã giải nén {n} ảnh về /content trong {time.time() - t0:.0f}s')
else:
    DATA = SYM / 'DISFA'                 # đọc thẳng Drive: chậm và hay lỗi FileNotFoundError
print('DATA =', DATA)

os.chdir(CODE)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
(CODE / 'checkpoints').mkdir(exist_ok=True)
w_src = SYM / 'Contrastive_V1' / 'checkpoints' / 'resnet50-19c8e357.pth'
w_dst = CODE / 'checkpoints' / 'resnet50-19c8e357.pth'
if w_src.exists() and not w_dst.exists():
    shutil.copy(w_src, w_dst)
import torch, torchvision
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| CPU', os.cpu_count(),
      '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ')

import re
def run_logged(cmd, name):
    """Chạy lệnh con, ghi toàn bộ output vào MyDrive/Sym/logs/ và in các dòng đáng xem.
    (Colab không hiện stdout của tiến trình con nếu chỉ dùng subprocess.run.)"""
    LOGS.mkdir(exist_ok=True)
    log = LOGS / f'{time.strftime("%Y%m%d_%H%M%S")}_{name}.log'
    print(f'>> {name} | log: {log.name}', flush=True)
    with open(log, 'w') as f, subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE,
                                               stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
        for line in p.stdout:
            f.write(line); f.flush()
            if line.startswith('{"epoch"'):
                r = json.loads(line)
                print(f"   ep{r['epoch']:>2}  wa={r['train']['wa']:.4f}  valF1={r['val']['macro_f1']:.3f}"
                      f"  {r['seconds'] / 60:.0f} phút", flush=True)
            elif line.startswith('epoch ') and ' step ' in line:
                if re.search(r'step (1000|2000)/', line):
                    print('   ' + line.rstrip(), flush=True)
            elif 'FutureWarning' not in line and 'warnings.warn' not in line:
                print('   ' + line.rstrip(), flush=True)
    if p.returncode:
        raise RuntimeError(f'{name} dừng với mã {p.returncode}, xem log {log}')

In [ ]:
# ==== Cell I: nhập run train trên máy GPU riêng (MyDrive/Sym/upload/aug1_*.zip) vào runs_aug1 — kiểm khoá, không ghi đè ====
import re, zipfile
sys.path.insert(0, str(CODE))
from stage2_decoder.contrast_ci import LOCK, run_problems

UP, BUSY_MIN, EPOCHS = SYM / 'upload', 15, 20
assert RUNS.name == 'runs_aug1', f'RUNS phải là runs_aug1, đang là {RUNS}'

def ok(run, v, k, s):
    return not run_problems(run, v, k, s, LOCK) and (run / 'last.pt').exists()

def park(z, sub):
    (UP / sub).mkdir(parents=True, exist_ok=True)
    shutil.move(str(z), str(UP / sub / z.name))

zips = sorted(UP.glob('aug1_*.zip'))
print(f'{len(zips)} zip chờ nhập trong {UP}', flush=True)
for z in zips:
    m = re.fullmatch(r'aug1_(baseline|full)_fold([123])_seed([012])', z.stem)
    if not m:
        print(f'  {z.name}: tên lạ, bỏ qua'); continue
    v, k, s = m[1], int(m[2]), int(m[3])
    target, prefix = RUNS / v / f'fold{k}' / f'seed{s}', f'{v}/fold{k}/seed{s}/'
    work = Path('/content/pc_import') / z.stem
    shutil.rmtree(work, ignore_errors=True)
    work.mkdir(parents=True)
    subprocess.run(['cp', str(z), str(work / 'run.zip')], check=True, timeout=1800)     # đọc Drive có timeout
    with zipfile.ZipFile(work / 'run.zip') as f:
        names = f.namelist()
        if any(not n.startswith(prefix) or '..' in n.split('/') for n in names):
            print(f'  {z.name}: zip chứa đường dẫn lạ → KHÔNG nhập'); park(z, 'rejected'); continue
        f.extractall(work / 'x')
    run = work / 'x' / v / f'fold{k}' / f'seed{s}'
    if not ok(run, v, k, s):
        print(f'  {z.name}: lệch khoá hoặc chưa xong {run_problems(run, v, k, s, LOCK)} → KHÔNG nhập')
        park(z, 'rejected'); continue
    gpu = json.loads((run / 'environment.json').read_text()).get('gpu')
    if (target / 'config.json').exists():
        if ok(target, v, k, s):
            print(f'  {z.name}: Drive đã có run này đủ {EPOCHS} epoch → giữ bản Drive, không ghi đè')
            park(z, 'duplicate'); continue
        files = [p for p in target.rglob('*') if p.is_file()]
        age = (time.time() - max(p.stat().st_mtime for p in files)) / 60 if files else 1e9
        if age < BUSY_MIN:
            print(f'  {z.name}: run này vừa được ghi {age:.0f} phút trước (Colab có thể đang train) → để lại, chạy lại cell sau')
            continue
        dest = SYM / 'failed_runs' / f'aug1_{v}_fold{k}_seed{s}_{time.strftime("%Y%m%d_%H%M%S")}_truoc_nhap'
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(target), str(dest))
        print(f'  bản dở trên Drive → failed_runs/{dest.name}')
    partial = target.with_name(target.name + '.partial')
    shutil.rmtree(partial, ignore_errors=True)
    partial.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['cp', '-r', str(run), str(partial)], check=True, timeout=3600)
    partial.rename(target)
    assert ok(target, v, k, s), f'{z.name}: sau khi chép lên Drive lại lệch, gửi Claude xem'
    park(z, 'done')
    shutil.rmtree(work)
    print(f'  {z.name}: ĐÃ NHẬP (GPU {gpu})', flush=True)

done = sum(ok(RUNS / v / f'fold{k}' / f'seed{s}', v, k, s) for v in ('baseline', 'full') for k in (1, 2, 3) for s in (0, 1, 2))
print(f'\nruns_aug1 hiện có {done}/18 run đủ {EPOCHS} epoch đúng khoá. Đang chờ Drive đồng bộ...', flush=True)
drive.flush_and_unmount()